# 04b — 5G Coverage & Adoption (honest rebuild)
SpiriCom · Huawei Technologies Tunisia · PFE 2026

Produces `data/outputs/coverage_5g.json` for the dashboard's 5G NETWORK COVERAGE
section (`Coverage5GSection.jsx`, served by `GET /api/coverage/5g`).

## Why the old payload was withdrawn
The v1 payload computed "5G adoption" from `traffic_5g` — a column that is
**91.8% median-imputed** (NB03b forensics). It displayed the data-collection
schedule, not adoption.

## Honest definitions used here
| Metric | Source | Why it is trustworthy |
|--------|--------|----------------------|
| 5G-capable device | `generation` contains `NR` | device attribute, never null, 6 clean categories |
| Using 5G now | `traffic_5g_imputed == 0` AND `traffic_5g > 0` | only the ~8% genuinely observed values |
| Province adoption | NR-capable share per `layer2name` | capability is observable for 100% of rows; real-usage counts (~17/province) would be statistically meaningless |
| 5G vs 4G performance | NR-capable vs LTE-only, NaN-aware means on **de-imputed** QoS | imputed QoS rows restored to NaN before averaging (NB04-2 pattern) |
| Churn rate | label v6, labelled population only | disengagement segmentation — same guardrails as everywhere |


In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

PROC_DIR, OUT_DIR = Path('data/processed'), Path('data/outputs')

df = pd.read_parquet(PROC_DIR / 'churn_labelled_v6.parquet')
print(f'{len(df):,} rows x {df.shape[1]} cols')

# De-impute QoS for honest averaging (same pattern as NB04-2).
# Only e2e_delay_ms and client_rtt_ms have *_imputed flags from NB03b;
# server_packet_loss_rate was not in IMPUTE_CHECK_COLS so no flag exists —
# its values may still contain NB00 median-imputed rows. Documented as a
# known limitation in the JSON data_quality block (NB04b-FIX-3).
for base in ['e2e_delay_ms', 'client_rtt_ms']:
    flag = f'{base}_imputed'
    if flag in df.columns:
        df.loc[df[flag] == 1, base] = np.nan

gen = df['generation'].fillna('UNKNOWN').astype(str).str.upper()
df['has_nr']  = gen.str.contains('NR').astype(int)
df['has_lte'] = gen.str.contains('LTE').astype(int)

# NB04b-FIX-4: if traffic_5g_imputed is absent (e.g. before NB03b runs),
# the fallback Series of 1s makes real_5g all-False → uses_5g = 0
# (safe default: assume all imputed, report 0 genuine 5G users).
real_5g = (df.get('traffic_5g_imputed', pd.Series(1, index=df.index)) == 0)
df['uses_5g'] = (real_5g & (df['traffic_5g'] > 0)).astype(int)
print(f'NR-capable : {int(df.has_nr.sum()):,} ({df.has_nr.mean()*100:.1f}%)')
print(f'Real traffic_5g observations: {int(real_5g.sum()):,} '
      f'-> using 5G now: {int(df.uses_5g.sum()):,}')


4,896 rows x 34 cols
NR-capable : 596 (12.2%)
Real traffic_5g observations: 400 -> using 5G now: 400


## 1 — KPI block

In [2]:
total = int(len(df))
using = int(df['uses_5g'].sum())
real_vals = df.loc[df['uses_5g'] == 1, 'traffic_5g']

kpi = {
    'total_subscribers'   : total,
    'adoption_rate_pct'   : round(using / total * 100, 1),
    'capable_devices_pct' : round(float(df['has_nr'].mean()) * 100, 1),
    'subscribers_using_5g': using,
    'subscribers_no_5g'   : total - using,
    'avg_5g_traffic'      : round(float(real_vals.mean()) / 1e6, 1)
                            if len(real_vals) else 0,   # MB, observed only
}
print(json.dumps(kpi, indent=2))


{
  "total_subscribers": 4896,
  "adoption_rate_pct": 8.2,
  "capable_devices_pct": 12.2,
  "subscribers_using_5g": 400,
  "subscribers_no_5g": 4496,
  "avg_5g_traffic": 222.9
}


## 2 — Adoption by province (NR-capable share per governorate)

In [3]:
g = (df.groupby(df['layer2name'].fillna('UNKNOWN'))
       .agg(total=('msisdn', 'count'),
            capable=('has_nr', 'sum'),
            churn_rate=('churn', 'mean')))      # NaN labels auto-excluded
g['ratio_5g_pct'] = (g['capable'] / g['total'] * 100).round(1)
g = g[g['total'] >= 10].sort_values('ratio_5g_pct', ascending=False)

by_province = [
    {'province': str(p), 'ratio_5g_pct': float(r['ratio_5g_pct']),
     'total': int(r['total']),
     'churn_rate': round(float(r['churn_rate']), 4)
                   if pd.notna(r['churn_rate']) else None}
    for p, r in g.iterrows()
]
print(f'{len(by_province)} provinces')
print(pd.DataFrame(by_province).head(8).to_string(index=False))


24 provinces
 province  ratio_5g_pct  total  churn_rate
   ARIANA          20.1    154      0.2740
    TUNIS          17.4    826      0.3448
 MEDENINE          16.7     90      0.3519
    GAFSA          15.8    101      0.3036
     SFAX          15.7    356      0.2857
   SOUSSE          15.3    346      0.3221
   KEBILI          15.0     40      0.4762
BEN AROUS          13.9    310      0.3293


## 3 — Generation mix

In [4]:
mix = gen.value_counts()
generation_mix = [
    {'generation': str(k), 'count': int(v),
     'pct': round(float(v) / total * 100, 1)}
    for k, v in mix.items()
]
print(pd.DataFrame(generation_mix).to_string(index=False))


  generation  count  pct
   2G/3G/LTE   3346 68.3
          2G    850 17.4
2G/3G/LTE/NR    596 12.2
       2G/3G     84  1.7
      2G/LTE     18  0.4
      3G/LTE      2  0.0


## 4 — Performance: 5G-capable vs LTE-only
NaN-aware means on de-imputed QoS. `avg_pkt_loss` stays a fraction
(the component multiplies by 100).

In [5]:
grp_5g  = df[df['has_nr'] == 1]
grp_4g  = df[(df['has_nr'] == 0) & (df['has_lte'] == 1)]

# NB04b-FIX-2: NaN guard for QoS means.
# After de-imputation, e2e_delay_ms and client_rtt_ms have NaN for 12-13%
# of rows. pandas .mean() skips NaN (correct), but if an entire group were
# all-NaN, float(NaN) propagates and json.dumps raises ValueError; or
# default=str would silently write the string "nan" instead of null,
# causing the dashboard JSX to receive a string where it expects a number.
# _safe() returns None (→ JSON null) whenever the mean is NaN.
def _safe(x, digits):
    return round(float(x), digits) if pd.notna(x) else None

def perf(grp):
    lab = grp['churn'].dropna()
    return {
        'count'       : int(len(grp)),
        'avg_latency' : _safe(grp['e2e_delay_ms'].mean(), 1),
        'avg_rtt'     : _safe(grp['client_rtt_ms'].mean(), 1),
        'avg_pkt_loss': _safe(grp['server_packet_loss_rate'].mean(), 5),
        'churn_rate'  : _safe(lab.mean(), 4) if len(lab) else None,
        'labelled'    : int(len(lab)),
    }

performance = {'mostly_5g': perf(grp_5g), 'mostly_4g': perf(grp_4g)}
print(json.dumps(performance, indent=2))
if (performance['mostly_5g']['churn_rate'] is not None
        and performance['mostly_4g']['churn_rate'] is not None):
    diff = (performance['mostly_5g']['churn_rate']
            - performance['mostly_4g']['churn_rate']) * 100
    print(f'\nDisengagement gap (5G-capable - LTE-only): {diff:+.1f} pts')


{
  "mostly_5g": {
    "count": 596,
    "avg_latency": 248.2,
    "avg_rtt": 157.0,
    "avg_pkt_loss": 0.00183,
    "churn_rate": 0.2833,
    "labelled": 420
  },
  "mostly_4g": {
    "count": 3366,
    "avg_latency": 306.3,
    "avg_rtt": 224.3,
    "avg_pkt_loss": 0.00141,
    "churn_rate": 0.3043,
    "labelled": 1972
  }
}

Disengagement gap (5G-capable - LTE-only): -2.1 pts


## 5 — Coverage gap alerts
Provinces with NR-capable share < 15% **and** disengagement above the overall
labelled rate — the underserved-region shortlist.

In [6]:
overall_rate = float(df['churn'].dropna().mean())
coverage_gaps = [
    p for p in by_province
    if p['ratio_5g_pct'] < 15
    and p['churn_rate'] is not None
    and p['churn_rate'] > overall_rate
]
coverage_gaps = sorted(coverage_gaps, key=lambda p: p['ratio_5g_pct'])
print(f'Overall disengagement (labelled): {overall_rate*100:.1f}%')
print(f'Coverage gaps: {len(coverage_gaps)}')
for p in coverage_gaps:
    print(f"  {p['province']:<14s} capable {p['ratio_5g_pct']:>5.1f}% | "
          f"disengaged {p['churn_rate']*100:.1f}%")


Overall disengagement (labelled): 33.8%
Coverage gaps: 9
  KAIROUAN       capable   6.1% | disengaged 42.4%
  KASSERINE      capable   8.1% | disengaged 37.5%
  MONASTIR       capable   8.6% | disengaged 37.2%
  JENDOUBA       capable   8.6% | disengaged 37.9%
  MANOUBA        capable   9.6% | disengaged 41.8%
  BIZERTE        capable  10.8% | disengaged 41.2%
  KEF            capable  11.3% | disengaged 39.5%
  MAHDIA         capable  12.8% | disengaged 46.8%
  TOZEUR         capable  13.0% | disengaged 45.5%


## 6 — Assemble & export `coverage_5g.json`

In [7]:
# NB04b-FIX-5: renamed engaged_churners → winback_targets.
# 'engaged_churners' was a contradiction: these customers ARE in the
# disengaged segment (churn == 1.0), not 'engaged'. The correct framing
# is 'NR-capable customers who are already disengaged' — the highest-value
# retention target because a 5G quality improvement may reactivate them.
ec = df[(df['has_nr'] == 1) & (df['churn'] == 1.0)]
winback_targets = {
    'count': int(len(ec)),
    'note' : 'NR-capable devices in the disengaged segment — '
             'prime 5G win-back targets: already capable, not yet using 5G',
}

payload = {
    'generated_at'  : datetime.now().isoformat(),
    'source'        : 'NB04b - churn_labelled_v6.parquet (label v6)',
    'definitions'   : {
        'adoption'  : 'subscribers with real (non-imputed) traffic_5g > 0',
        'capability': "device generation contains 'NR'",
        'province_metric': 'NR-capable share per layer2name (capability, '
                           'not usage - usage observed for only 8.2% of rows)',
        'performance_groups': 'mostly_5g = NR-capable, mostly_4g = LTE-only',
    },
    'data_quality'  : {
        'traffic_5g_imputed_pct': 91.8,
        # NB04b-FIX-3: server_packet_loss_rate is used in perf() but was
        # NOT in NB03b's IMPUTE_CHECK_COLS, so no *_imputed flag was created
        # for it. Its values may include NB00 median-imputed rows that could
        # not be retroactively detected. Performance comparison values for
        # avg_pkt_loss should be interpreted with this caveat until the NB00
        # semantic-imputation fix is re-run and all QoS flags are rebuilt.
        'qos_not_deimputed'      : ['server_packet_loss_rate'],
        'note': 'usage-based metrics computed on the observed subset only; '
                're-run after the NB00 semantic-imputation fix for full '
                'population coverage and complete QoS de-imputation',
    },
    'kpi'           : kpi,
    'by_province'   : by_province,
    'generation_mix': generation_mix,
    'performance'   : performance,
    'coverage_gaps' : coverage_gaps,
    'winback_targets': winback_targets,
}

out = OUT_DIR / 'coverage_5g.json'
with open(out, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, default=str)
print(f'Saved: {out}')
print(f"kpi: {payload['kpi']}")
print(f"provinces: {len(by_province)} | gaps: {len(coverage_gaps)} | "
      f"win-back targets: {winback_targets['count']}")
print('\nNext -> register coverage_api.py (or point the analytics_api')
print('endpoint at this file via artifact_cache.get_json).')


Saved: data\outputs\coverage_5g.json
kpi: {'total_subscribers': 4896, 'adoption_rate_pct': 8.2, 'capable_devices_pct': 12.2, 'subscribers_using_5g': 400, 'subscribers_no_5g': 4496, 'avg_5g_traffic': 222.9}
provinces: 24 | gaps: 9 | win-back targets: 119

Next -> register coverage_api.py (or point the analytics_api
endpoint at this file via artifact_cache.get_json).
